| 모델 | 역할 |
|---|---|
| `gpt-4o-mini` (OpenAI) | 클라우드 API 기반 구조화된 출력 (Responses API) |
| `exaone3.5` (Ollama, 로컬) | 로컬 실행 모델의 구조화된 출력 |

### Chat Completions API → Responses API, 무엇이 바뀌었나

| 항목 | Chat Completions (이전) | Responses (이번) |
|---|---|---|
| 엔드포인트 | `/v1/chat/completions` | `/v1/responses` |
| 호출 메서드 | `client.chat.completions.create` / `.beta.chat.completions.parse` | `client.responses.create` / `.responses.parse` |
| 입력 구조 | `messages=[{"role":...,"content":...}, ...]` | `instructions="..."` (시스템 역할) + `input="..."` (사용자 입력) |
| 스키마 지정 위치 | `response_format={"type":"json_schema", "json_schema":{...}}` | `text={"format": {"type":"json_schema", ...}}` |
| Pydantic 연동 | `response_format=Model` | `text_format=Model` |
| 결과 텍스트 | `completion.choices[0].message.content` | **`response.output_text`** (헬퍼 프로퍼티) |
| 결과 파싱 객체 | `completion.choices[0].message.parsed` | **`response.output_parsed`** |
| 거부(refusal) | `message.refusal` (최상위 필드) | `output` 배열의 메시지 아이템 안 `content`에 `type:"refusal"` 아이템으로 등장 |
| Tool 스키마 | 중첩형 `{"type":"function","function":{"name":...,"parameters":...}}` | **평탄화**: `{"type":"function","name":...,"parameters":...}` |
| 스트리밍 완료 | `stream.get_final_completion()` | `stream.get_final_response()` |

Ollama도 v0.13.3부터 `/v1/responses`를 지원하기 시작해서(비상태 모드), 이번 버전에서는
**OpenAI와 Ollama 양쪽 모두 Responses API로 통일**할 수 있습니다.

### 이 노트북에서 다루는 케이스 (총 14가지, 이전과 동일한 구성)

| # | 케이스 | 핵심 개념 |
|---|---|---|
| A | 레거시 JSON 모드 | `text={"format":{"type":"json_object"}}` |
| B | 수동 JSON Schema | dict로 직접 스키마 작성, `strict:true` |
| C | Pydantic + `.parse()` | `text_format=Model`, `output_parsed` |
| D | 중첩 구조 | 리스트 안에 객체 (성적표) |
| E | Enum 분류 | 감성 분석 (긍정/부정/중립) |
| F | Optional/Nullable | 없을 수도 있는 필드 처리 |
| G | Union(anyOf) | 서로 다른 스키마 중 하나를 판별 |
| H | Refusal 처리 | `output` 배열에서 refusal 콘텐츠 아이템 확인 |
| I | 스트리밍 | `response.output_text.delta` 이벤트 |
| J | Function/Tool Calling | 평탄화된 tool 스키마, `parsed_arguments` |
| K | Ollama 네이티브 | `format=schema` 로컬 모델 구조화된 출력 |
| L | Ollama Responses 호환 | 같은 Responses API 코드로 로컬 모델 접근 |
| M | 검증-재시도 패턴 | 실패를 감지하고 스스로 고쳐 재요청 |
| N | 두 모델 비교 | 같은 입력, 같은 스키마로 나란히 비교 |

### 1. 이론 — 구조화된 출력은 왜 필요하고, 어떻게 진화했나

### 문제의 시작: "그냥 JSON으로 답해줘"
프롬프트에 "JSON 형식으로 답변해줘"라고 적어서 응답을 받으면, 코드블록으로 감싸서 반환하거나\
필드 이름이 매번 미묘하게 다르거나, 숫자를 문자열로 반환하는 등 사고가 종종 발생합니다.\
이 문제는 **"출력이 특정 문법/스키마를 따르도록 강제할 수 있는가"**의 문제입니다.

### 1단계 → 2단계 → 3단계

| 단계 | 방식 | 보장 수준 |
|---|---|---|
| 1단계 | 프롬프트로 부탁 ("JSON으로 답해줘") | 아무것도 보장 안 됨 |
| 2단계 | **JSON 모드** (`text={"format":{"type":"json_object"}}`) | 문법적으로 유효한 JSON은 보장 (필드 구성은 미보장) |
| 3단계 | **JSON Schema / Structured Outputs** (`text={"format":{"type":"json_schema",...}}`) | 스키마(필드명, 타입, 필수 여부)까지 보장 |

3단계는 "제약된 디코딩(Constrained Decoding)"이라는 기법으로 동작합니다.\
모델이 다음 토큰을 고를 때, 애초에 스키마를 위반하는 토큰은 후보에서 제외해버리는 방식입니다.

### 왜 Responses API인가
OpenAI는 Chat Completions API를 계속 지원하지만, **Responses API를 새로운 표준 인터페이스**로\
밀고 있습니다. `output_text` 헬퍼로 텍스트를 바로 꺼낼 수 있고, `previous_response_id`로\
대화 상태를 OpenAI 서버에 맡길 수 있으며(이 노트북에서는 단일 턴 예제라 사용하지 않습니다),\
도구 호출 스키마가 더 평탄해져서 다루기 쉬워졌습니다.

### 대안 경로: Function(Tool) Calling
`tool_choice`로 특정 함수 호출을 강제하면, "함수의 인자값"이라는 형태로 구조화된 데이터를\
받아낼 수 있습니다. Case J에서 다룹니다.

### 로컬 모델(EXAONE 등)은 다르다
로컬에서 Ollama로 돌리는 EXAONE 3.5는 **Ollama 자체의 `format` 파라미터**로 유사한 효과를 냅니다.\
Ollama는 v0.13.3부터 `/v1/responses` 엔드포인트도 지원하기 시작했으므로(단, 대화 상태 관리 기능\
제외), OpenAI용으로 짠 Responses API 코드를 거의 그대로 재사용할 수 있습니다 (Case L).


### 2. 환경 설정


In [ ]:
#%pip install -q openai ollama pydantic
#print("설치 완료")


In [ ]:
import os
import json
from enum import Enum
from typing import Optional, Union, Literal

from pydantic import BaseModel, ValidationError
import openai
from openai import OpenAI
import ollama

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL = "gpt-4o-mini"
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

OLLAMA_MODEL = "exaone3.5"


def need_openai() -> bool:
    """OPENAI_API_KEY가 없으면 셀 실행을 건너뛰고 안내만 출력합니다."""
    if not OPENAI_API_KEY:
        print("[건너뜀] OPENAI_API_KEY가 설정되지 않아 이 셀은 실행하지 않습니다.")
        return False
    return True


def need_ollama() -> bool:
    """Ollama 서버에 연결할 수 없으면 셀 실행을 건너뛰고 안내만 출력합니다."""
    try:
        ollama.list()
        return True
    except Exception as e:
        print(f"[건너뜀] Ollama 서버에 연결할 수 없습니다 ({e}).")
        print(f"        'ollama serve' 실행 및 'ollama pull {OLLAMA_MODEL}' 확인 후 다시 실행하세요.")
        return False


print("OpenAI API 키:", "감지됨" if OPENAI_API_KEY else "없음 -> OpenAI 예제는 건너뜁니다")
print("Ollama 연결:", "가능" if need_ollama() else "불가 -> Ollama 예제는 건너뜁니다")


---
### Case A — 레거시 JSON 모드 (`json_object`)

Responses API에서는 `text={"format": {"type": "json_object"}}` 로 지정합니다.\
Chat Completions의 `response_format={"type":"json_object"}` 와 보장 수준은 동일합니다\
(문법만 보장, 필드 구성은 미보장).

결과 텍스트는 `response.output_text` 헬퍼로 바로 꺼낼 수 있습니다\
(Chat Completions의 `completion.choices[0].message.content` 보다 짧습니다).


In [ ]:
if need_openai():
    prompt = (
        "다음 리뷰에서 상품명과 평점(1~5 정수)을 JSON으로 추출해줘. "
        '형식: {"product": "...", "rating": ...}\n\n'
        "리뷰: 노트북 정말 좋아요! 배터리도 오래가고 화면도 선명해요. 5점 만점에 5점 줍니다."
    )
    response = openai_client.responses.create(
        model=OPENAI_MODEL,
        instructions="사용자가 요청한 정보를 JSON으로만 응답하세요.",
        input=prompt,
        text={"format": {"type": "json_object"}},
    )
    raw = response.output_text
    print("원본 응답 문자열:", raw)
    data = json.loads(raw)  # 문법은 항상 보장되므로 파싱 자체는 실패하지 않음
    print("파싱 결과:", data)
    print("주의: 'product', 'rating' 필드가 실제로 존재하는지는 우리가 직접 확인해야 합니다.")


---
### Case B — 수동으로 작성한 JSON Schema (Strict 모드)

Pydantic 없이, JSON Schema를 dict로 직접 작성해서 `text.format`에 넣는 방식입니다.\
`json_schema` 포맷 dict에는 `name`, `schema`, `strict` 세 키가 필요합니다\
(Chat Completions에서는 이 세 키가 `json_schema` 아래 한 번 더 감싸져 있었는데,\
Responses API에서는 `format` 바로 아래로 평탄화되었습니다).


In [ ]:
if need_openai():
    manual_schema = {
        "type": "object",
        "properties": {
            "product": {"type": "string"},
            "rating": {"type": "integer"},
        },
        "required": ["product", "rating"],
        "additionalProperties": False,
    }
    response = openai_client.responses.create(
        model=OPENAI_MODEL,
        instructions="리뷰에서 상품명과 평점을 추출하세요.",
        input="노트북 정말 좋아요! 5점 만점에 5점 줍니다.",
        text={
            "format": {
                "type": "json_schema",
                "name": "review_extraction",
                "schema": manual_schema,
                "strict": True,
            }
        },
    )
    raw = response.output_text
    print("원본 응답:", raw)
    print("파싱 결과:", json.loads(raw))


---
### Case C — Pydantic + `.parse()` (실무 권장 방식)

`client.responses.parse(text_format=PydanticModel클래스)` 를 쓰면\
`response.output_parsed` 에 **타입이 있는 파이썬 객체**가 바로 담깁니다.\
이후 케이스들은 대부분 이 방식을 기본으로 사용합니다.


In [ ]:
class ReviewExtraction(BaseModel):
    product: str
    rating: int


if need_openai():
    response = openai_client.responses.parse(
        model=OPENAI_MODEL,
        instructions="리뷰에서 상품명과 평점을 추출하세요.",
        input="노트북 정말 좋아요! 5점 만점에 5점 줍니다.",
        text_format=ReviewExtraction,
    )
    result = response.output_parsed
    print("타입:", type(result))
    print("객체:", result)
    print("상품명:", result.product, "| 평점:", result.rating)


---
### Case D — 중첩 구조 (리스트 안에 객체)

학생 한 명의 성적표처럼 **객체 안에 리스트, 리스트 안에 또 다른 객체**가 들어가는 경우입니다.\
스키마 정의(Pydantic 모델)는 Chat Completions 때와 **완전히 동일**합니다 — 달라지는 건\
호출 메서드와 결과를 꺼내는 방식뿐입니다.


In [ ]:
class Subject(BaseModel):
    name: str
    score: int


class StudentReport(BaseModel):
    student_name: str
    grade_level: int
    subjects: list[Subject]
    average_score: float


if need_openai():
    text = "둘리 학생은 3학년이고, 국어 88점, 수학 95점, 영어 76점을 받았습니다."
    response = openai_client.responses.parse(
        model=OPENAI_MODEL,
        instructions=(
            "학생 성적 정보를 구조화해서 추출하세요. "
            "average_score는 과목 점수들의 평균을 직접 계산해서 채우세요."
        ),
        input=text,
        text_format=StudentReport,
    )
    report = response.output_parsed
    print(f"{report.student_name} ({report.grade_level}학년) - 평균 {report.average_score}점")
    for s in report.subjects:
        print(f"  - {s.name}: {s.score}점")


---
### Case E — Enum 기반 분류 (감성 분석)

값이 정해진 몇 가지 카테고리 중 하나여야 한다면 자유 문자열보다 **Enum**을 쓰는 게 훨씬 안전합니다.


In [ ]:
class Sentiment(str, Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "중립"


class SentimentResult(BaseModel):
    sentiment: Sentiment
    confidence: float
    reason: str


reviews = [
    "배송이 너무 느리고 포장도 엉망이었어요.",
    "그냥 무난했어요. 특별한 건 없네요.",
    "가격 대비 최고의 제품입니다!",
]

if need_openai():
    for review in reviews:
        response = openai_client.responses.parse(
            model=OPENAI_MODEL,
            instructions="리뷰의 감성을 분석하세요.",
            input=review,
            text_format=SentimentResult,
        )
        r = response.output_parsed
        print(f"[{r.sentiment.value}] (확신도 {r.confidence:.2f}) {review}")
        print(f"    -> {r.reason}")


---
### Case F — Optional(Nullable) 필드

`Optional[str] = None` 으로 선언하면 자동으로 `anyOf: [string, null]` 타입으로 변환되는 것은\
Chat Completions 때와 동일합니다. strict 모드에서는 모든 필드가 항상 `required`이므로,\
"필드 자체가 없어짐"이 아니라 "값이 `null`일 수 있음"이라는 점을 기억하세요.


In [ ]:
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None  # 없으면 모델이 null을 채워 넣음


texts = [
    "저는 이수진이고 이메일은 sujin@example.com 입니다.",
    "박현우입니다. 이메일은 hyunwoo@example.com, 전화번호는 010-1234-5678이에요.",
]

if need_openai():
    for t in texts:
        response = openai_client.responses.parse(
            model=OPENAI_MODEL,
            instructions="연락처 정보를 추출하세요. 전화번호가 텍스트에 없으면 null로 두세요.",
            input=t,
            text_format=ContactInfo,
        )
        print(response.output_parsed)


---
### Case G — Union(anyOf): 서로 다른 스키마 중 하나 판별하기

`Literal["meeting"]` / `Literal["vacation"]` 처럼 판별 필드(discriminator)를 각 하위 모델에\
넣어두면, `entry.event_type` 값만 보고도 어떤 타입인지 안전하게 분기할 수 있습니다.


In [ ]:
class MeetingEvent(BaseModel):
    event_type: Literal["meeting"]
    title: str
    attendees: list[str]


class VacationEvent(BaseModel):
    event_type: Literal["vacation"]
    person: str
    days: int


class CalendarEntry(BaseModel):
    entry: Union[MeetingEvent, VacationEvent]


texts = [
    "다음 주 월요일에 둘리, 또치와 프로젝트 회의가 있어요.",
    "민지가 3일간 휴가를 냈습니다.",
]

if need_openai():
    for t in texts:
        response = openai_client.responses.parse(
            model=OPENAI_MODEL,
            instructions="일정 텍스트를 회의 또는 휴가 이벤트로 분류해서 구조화하세요.",
            input=t,
            text_format=CalendarEntry,
        )
        entry = response.output_parsed.entry
        if isinstance(entry, MeetingEvent):
            print(f"[회의] {entry.title} (참석자: {', '.join(entry.attendees)})")
        else:
            print(f"[휴가] {entry.person} - {entry.days}일")


---
### Case H — Refusal(응답 거부) 처리

Chat Completions에서는 `message.refusal`이라는 최상위 필드로 거부 여부를 확인했지만,\
Responses API에서는 **`output` 배열 안 메시지 아이템의 `content` 리스트에 `type:"refusal"`인\
아이템**으로 등장합니다. 구조가 한 단계 더 들어가 있으니, 헬퍼 함수로 감싸서 확인하는 것이 편합니다.\
(아래 예제 자체는 무해한 리뷰라 실제로 거부되지는 않지만, 프로덕션 코드에서는 이 체크가 필수입니다.)


In [ ]:
def extract_refusal(response) -> str | None:
    """Responses API 결과에서 refusal 텍스트를 찾아 반환합니다. 없으면 None."""
    for item in response.output:
        if item.type == "message":
            for content_item in item.content:
                if content_item.type == "refusal":
                    return content_item.refusal
    return None


if need_openai():
    response = openai_client.responses.parse(
        model=OPENAI_MODEL,
        instructions="리뷰의 감성을 분석하세요.",
        input="이 카페 분위기가 정말 좋아요.",
        text_format=SentimentResult,
    )

    refusal = extract_refusal(response)
    if refusal:
        print("모델이 응답을 거부했습니다:", refusal)
    else:
        print("정상 파싱 결과:", response.output_parsed)


---
### Case I — 스트리밍 + 구조화된 출력

`client.responses.stream()` 컨텍스트 매니저를 사용합니다. 이벤트 타입 이름도 바뀌어서,\
텍스트 조각은 **`response.output_text.delta`** 이벤트로 옵니다 (Chat Completions의\
`content.delta`에 대응). 완료 시에는 `stream.get_final_response()`로 최종 결과를 받습니다.


In [ ]:
if need_openai():
    with openai_client.responses.stream(
        model=OPENAI_MODEL,
        instructions="리뷰의 감성을 분석하세요.",
        input="배송도 빠르고 품질도 훌륭해요. 재구매 의사 100%입니다.",
        text_format=SentimentResult,
    ) as stream:
        for event in stream:
            if event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
        final = stream.get_final_response()

    print()  # 줄바꿈
    print("최종 파싱 결과:", final.output_parsed)


---
### Case J — Function(Tool) Calling으로 구조화된 출력 얻기

`openai.pydantic_function_tool()`로 도구를 만드는 코드는 Chat Completions 때와 **동일**합니다.\
SDK가 내부적으로 이 도구가 Pydantic 기반인 것을 인식해서, Responses API의 평탄화된 도구 스키마로\
자동 변환해줍니다. `tool_choice`만 평탄화된 형태(`{"type":"function","name":...}`, `function` 중첩 없음)로\
바뀌었습니다.

결과는 `response.output`에서 `type == "function_call"`인 아이템을 찾아 꺼내고,\
**`.parsed_arguments`**에 이미 Pydantic 객체로 검증된 값이 들어있습니다\
(Chat Completions처럼 `json.loads()` + `model_validate()`를 직접 할 필요가 없습니다).


In [ ]:
sentiment_tool = openai.pydantic_function_tool(
    SentimentResult,
    name="save_sentiment",
    description="분석된 감성 결과를 저장합니다.",
)

if need_openai():
    response = openai_client.responses.parse(
        model=OPENAI_MODEL,
        instructions="리뷰를 분석하고 save_sentiment 함수를 호출해 결과를 저장하세요.",
        input="직원분이 친절했지만 음식은 그저 그랬어요.",
        tools=[sentiment_tool],
        tool_choice={"type": "function", "name": "save_sentiment"},  # 무조건 이 함수를 호출하도록 강제
    )
    call_item = next(item for item in response.output if item.type == "function_call")
    print("호출된 함수:", call_item.name)
    print("구조화된 인자 (이미 파싱됨):", call_item.parsed_arguments)


---
### 3. 이론 — 로컬 모델(EXAONE 3.5)의 구조화된 출력은 무엇이 다른가

| | OpenAI (`gpt-4o-mini`) | Ollama (`exaone3.5`) |
|---|---|---|
| 스키마 전달 위치 (네이티브) | `text={"format": {"type":"json_schema", ...}}` | `format=<JSON Schema dict>` |
| Responses API 호환 | 정식 지원 | v0.13.3+ 지원 (비상태 모드만) |
| Pydantic 연동 | `text_format=PydanticModel` (SDK가 자동 변환+파싱) | `format=Model.model_json_schema()` 로 직접 전달, 응답은 `Model.model_validate_json()`으로 직접 파싱 |
| 신뢰도 | 100%에 가까움 | 모델/양자화 수준에 따라 달라짐 — 검증 로직이 더 중요해짐 |

즉, **로컬 모델을 쓸수록 Case M의 "검증-재시도 패턴"이 훨씬 중요해집니다.**


---
### Case K — Ollama 네이티브 방식 (EXAONE 3.5)

이 케이스는 OpenAI SDK가 아니라 **`ollama` 파이썬 라이브러리를 직접 사용**하므로,\
Responses API 전환과 무관하게 동일합니다. `format` 파라미터에 `Model.model_json_schema()`를\
그대로 넘기고, 응답은 `Model.model_validate_json()`으로 직접 파싱합니다.


In [ ]:
if need_ollama():
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": "다음 리뷰의 감성을 분석해줘: '가격 대비 훌륭한 성능입니다.' JSON으로만 답변해줘.",
            }
        ],
        format=SentimentResult.model_json_schema(),
        options={"temperature": 0},
    )
    result = SentimentResult.model_validate_json(response.message.content)
    print(result)


---
### Case L — Ollama의 Responses API 호환 엔드포인트 사용하기

Ollama는 v0.13.3부터 `/v1/responses`도 지원합니다 (대화 상태 관리 기능 제외). 즉, `base_url`만\
바꾸면 **OpenAI Responses API용으로 짠 코드를 그대로 재사용**할 수 있습니다.

다만 신뢰도가 중요한 실습에서는 여전히 **Case K(네이티브 방식)를 기본으로 추천**합니다.\
이 방식은 "여러 LLM 제공자를 같은 Responses 인터페이스로 다루고 싶을 때"의 참고용으로 알아두면 좋습니다.


In [ ]:
ollama_openai_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # 키는 무시되지만 형식상 필요

if need_ollama():
    response = ollama_openai_client.responses.create(
        model=OLLAMA_MODEL,
        input="다음 리뷰의 감성을 분석해줘: '배송이 너무 늦었어요.'",
        text={
            "format": {
                "type": "json_schema",
                "name": "sentiment_result",
                "schema": SentimentResult.model_json_schema(),
                "strict": True,
            }
        },
    )
    raw = response.output_text
    print("원본 응답:", raw)
    print("파싱 결과:", SentimentResult.model_validate_json(raw))


---
### Case M — 검증-재시도(Validate-Retry) 패턴

Structured Outputs가 없는 환경이나 Ollama 호환 레이어가 기대만큼 스키마를 지키지 않는 경우를\
대비해, 직접 검증하고 실패하면 스스로 고쳐서 재요청하는 유틸리티를 만들어봅니다.

이 부분은 실제 LLM 호출 없이도 동작을 100% 확인할 수 있도록, "불안정한 모델"을 흉내내는\
목업(mock) 함수로 실제 재시도 로직을 직접 실행해봅니다. (실제 서비스에서는 `mock_llm_call` 자리에\
`openai_client.responses.parse(...)` 또는 `ollama.chat(...)` 호출을 넣으면 됩니다.)


In [ ]:
class Person(BaseModel):
    name: str
    age: int


# 실제 LLM 대신, "불안정한 모델의 응답"을 순서대로 흉내내는 목업 함수
_mock_attempts = [
    '{"name": "홍길동", "age": "서른",}',   # 1차: JSON 문법 오류 (trailing comma) + 타입 오류
    '{"name": "홍길동", "age": "서른"}',    # 2차: 문법은 OK, 타입 오류 (age가 문자열)
    '{"name": "홍길동", "age": 30}',        # 3차: 정상
]
_attempt_state = {"n": 0}


def mock_llm_call(prompt: str) -> str:
    """실제 LLM 호출(예: openai_client.responses.parse) 대신 위 시나리오를 순서대로 반환합니다."""
    idx = min(_attempt_state["n"], len(_mock_attempts) - 1)
    _attempt_state["n"] += 1
    return _mock_attempts[idx]


def call_with_retry(schema_model, llm_call_fn, prompt: str, max_retries: int = 3):
    """LLM 호출 -> JSON 파싱 -> 스키마 검증을 시도하고, 실패하면 오류 내용을 프롬프트에
    덧붙여 재요청하는 범용 유틸리티.
    """
    last_error = None
    current_prompt = prompt

    for attempt in range(1, max_retries + 1):
        raw = llm_call_fn(current_prompt)

        try:
            parsed_json = json.loads(raw)
        except json.JSONDecodeError as e:
            last_error = f"JSON 문법 오류: {e}"
            print(f"[시도 {attempt}] JSON 파싱 실패 -> {last_error}")
            current_prompt = (
                f"{prompt}\n\n[이전 응답에 문제가 있었습니다] {last_error}\n"
                f"문제가 된 응답: {raw}\n올바른 JSON 문법으로 다시 응답해줘."
            )
            continue

        try:
            result = schema_model.model_validate(parsed_json)
            print(f"[시도 {attempt}] 검증 성공!")
            return result
        except ValidationError as e:
            last_error = str(e)
            print(f"[시도 {attempt}] 스키마 검증 실패")
            current_prompt = (
                f"{prompt}\n\n[이전 응답에 문제가 있었습니다] 스키마 검증 실패: {last_error}\n"
                f"문제가 된 응답: {raw}\n스키마에 맞게 다시 응답해줘."
            )

    raise RuntimeError(f"최대 재시도 횟수({max_retries}) 초과. 마지막 오류: {last_error}")


result = call_with_retry(Person, mock_llm_call, "사람 정보를 추출해줘")
print()
print("최종 결과:", result)


---
### Case N — GPT-4o-mini vs EXAONE 3.5 나란히 비교

같은 스키마(`SentimentResult`), 같은 입력 리뷰를 두 모델에 동시에 던져서 결과를 비교합니다.\
GPT-4o-mini 쪽은 Responses API(`responses.parse`), EXAONE 쪽은 Ollama 네이티브 방식을 사용합니다.


In [ ]:
if need_openai() and need_ollama():
    review = "생각보다 화면이 크고 선명해서 만족스러워요. 다만 배터리가 반나절 밖에 안 가는건 아쉽네요."

    gpt_response = openai_client.responses.parse(
        model=OPENAI_MODEL,
        instructions="리뷰의 감성을 분석하세요.",
        input=review,
        text_format=SentimentResult,
    )
    gpt_result = gpt_response.output_parsed

    ollama_response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": f"다음 리뷰의 감성을 분석해줘: \'{review}\'"}],
        format=SentimentResult.model_json_schema(),
        options={"temperature": 0},
    )
    ollama_result = SentimentResult.model_validate_json(ollama_response.message.content)

    print("리뷰:", review)
    print()
    print("GPT-4o-mini ->", gpt_result)
    print("EXAONE 3.5  ->", ollama_result)


---
### 4. 참고자료 — OpenAI Structured Outputs 제약사항 정리

Chat Completions든 Responses든 **스키마 자체의 제약사항은 동일**합니다. 
(둘 다 같은 StructuredOutputs 엔진을 사용합니다). 바뀌는 건 스키마를 감싸는 바깥쪽 파라미터 구조뿐입니다.

| 제약 | 내용 |
|---|---|
| 필수 필드 | 모든 필드가 `required`에 있어야 함. "선택적" 필드는 `anyOf: [type, null]` 로 표현 (Optional + 기본값 None) |
| additionalProperties | 모든 객체에 `"additionalProperties": false` 필요 (Pydantic 사용 시 SDK가 자동 처리) |
| 중첩 깊이 | 최대 5단계 |
| 객체 속성 개수 | 스키마 전체에서 최대 100개 |
| 문자열 총 길이 | 속성명 + enum 값 + const 값의 총 길이 15,000자 이내 |
| enum 값 개수 | 전체 최대 500개, 250개 초과 시 총 문자열 길이 7,500자 이내 |
| 미지원 키워드 | `minLength`, `maxLength`, `pattern`, `minimum`, `maximum` 등은 무시됨 |
| 재귀 스키마 | Pydantic으로 자기 자신을 참조하는 모델은 SDK 자동 변환에서 오류 발생 → 평평한 구조로 재설계 필요 |

### 방식 선택 가이드

- **정형화된 데이터 추출/분류가 목적** → Case C (`responses.parse` + `text_format`) 를 기본값으로 사용
- **다른 실제 도구(Tool)들과 함께 에이전트를 구성** → Case J (Function Calling)
- **긴 응답을 점진적으로 UI에 보여줘야 함** → Case I (스트리밍)
- **로컬/오픈소스 모델 사용** → Case K를 기본으로, 반드시 Case M(검증-재시도)을 함께 적용
- **여러 턴에 걸친 대화 상태를 서버에 맡기고 싶음** → Responses API의 `previous_response_id` 활용 (이 노트북에서는 단일 턴이라 다루지 않음)


---
## 정리 — 실무 체크리스트 (Responses API 기준)

1. **`response_format` → `text.format`, `messages` → `instructions`+`input`** — 이 세 가지 이름 변경이
   Chat Completions 코드를 Responses API로 옮길 때 가장 먼저 부딪히는 부분입니다.
2. **결과는 `output_text`/`output_parsed` 헬퍼로 꺼내세요.** `output` 배열을 직접 순회할 필요는
   거의 없습니다 (단, Refusal 체크나 Tool Call 추출처럼 배열 구조가 의미를 가질 때는 예외).
3. **Tool 스키마는 평탄화되었습니다.** `openai.pydantic_function_tool()`로 만드는 코드 자체는 그대로 두고,
   `tool_choice`만 `{"type":"function","name":...}` 형태로 바꿔주면 됩니다.
4. **`message.refusal` 체크가 `output` 배열 순회로 바뀌었습니다.** 헬퍼 함수로 감싸서 재사용하세요.
5. **로컬 모델(Ollama)을 쓸수록 검증-재시도 로직이 필수**입니다. Ollama v0.13.3+ 라면 Responses
   엔드포인트도 사용할 수 있지만, 스키마 준수율까지 OpenAI와 동일하게 보장되는 것은 아닙니다.
6. **Structured Outputs는 "형식"만 보장합니다. "내용"이 맞는지는 여전히 우리 책임**입니다.